# MCDI502 · Taller 1:Gestión de Datos y Tecnologías Big Data con Apache Spark
## Grupo 4 

**Asignatura:** MCDI502 · Gestión de Datos y Tecnologías Big Data
**Programa:** Magíster en Ciencias de Datos e Inteligencia Artificial, Universidad Andrés Bello
**Integrantes:** Felipe Díaz Toro, Pablo Ríos Passteni, Guillermo Subiabre Chacón, Ninoska Yévenes Hernández
**Herramientas:** Apache Spark 3.5.9 (PySpark)
**Datos:** modelo histórico de los Juegos Olímpicos (deportista, equipo, evento, resultado, juego)

## 1. Configuración inicial del entorno
### 1.1 Java y Spark, y variables de entorno

In [1]:
# Java 17 y PySpark 3.5.9 ya están instalados en el entorno de conda del grupo (grupo4_mcdi502)
# Se crean las variables de entorno necesarias: JAVA_HOME apunta al Java de ese entorno
import os
import sys

os.environ["JAVA_HOME"] = os.path.join(sys.prefix, "lib", "jvm")

print("✅ Variables de entorno configuradas")

✅ Variables de entorno configuradas


### 1.2 SparkSession y SparkContext

In [2]:
# Importar findspark para inicializar Spark
import findspark
findspark.init()

# Importar SparkSession (clase principal para trabajar con Spark)
from pyspark.sql import SparkSession

# Crear la sesión de Spark en modo local usando todos los núcleos disponibles
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('Taller1 Grupo4') \
        .getOrCreate()

# Crear el SparkContext (necesario para trabajar con RDDs)
sc = spark.sparkContext

print("✅ Sesión Spark creada")
print("Spark Version:", spark.version)

26/11/04 13:57:11 WARN Utils: Your hostname, MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.249 instead (on interface en0)
26/11/04 13:57:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 13:57:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Sesión Spark creada
Spark Version: 3.5.9


## 2. RDDs: creación y unión (ID 3.1)
### 2.0 Ruta de los datos 


In [9]:
# El notebook está en notebooks/, por eso los datos se buscan un nivel arriba
ruta = '../data/raw/taller1_sumativa2/'

### 2.1 RDD deportista (6 particiones)

In [10]:
# Cada elemento del RDD es una línea del archivo (deportista.csv no tiene encabezado)
deportista = sc.textFile(ruta + 'deportista.csv', 6)

print("Particiones de deportista:", deportista.getNumPartitions())
deportista.take(3)

Particiones de deportista: 6


['1,A Dijiang,1,24,180,80,199',
 '2,A Lamusi,1,23,170,60,199',
 '3,Gunnar Nielsen Aaby,1,24,0,0,273']

### 2.2 RDD deportista2

In [11]:
deportista2 = sc.textFile(ruta + 'deportista2.csv')

print("Particiones de deportista2:", deportista2.getNumPartitions())
deportista2.take(3)

Particiones de deportista2: 2


['67787,Lee BongJu,1,27,167,56,970',
 '67788,Lee BuTi,1,23,164,54,203',
 '67789,Anthony N. Buddy Lee,1,34,172,62,1096']

### 2.3 Unión de RDDs: deportistaTotal

In [12]:
# union() es una transformación: junta ambos RDDs sin eliminar duplicados
deportistaTotal = deportista.union(deportista2)

print("Particiones de deportistaTotal:", deportistaTotal.getNumPartitions())

Particiones de deportistaTotal: 8


### 2.4 Cantidad de registros de deportistaTotal

In [13]:
# count() es una acción: dispara el cálculo
print("Registros en deportistaTotal:", deportistaTotal.count())

Registros en deportistaTotal: 135571


### 2.5 Conversión a DataFrame deportista

In [14]:
from pyspark.sql.functions import col

columnas = ["deportista_id", "nombre", "genero", "edad", "altura", "peso", "equipo_id"]

# split(",")[:7]: una fila de deportista.csv (id 16587) trae una coma de más y tendría 8 campos
deportista = deportistaTotal.map(lambda linea: linea.split(",")[:7]).toDF(columnas)

# Todo llega como texto: se convierte a tipo numérico (peso a double por sus decimales)
deportista = deportista.withColumn("deportista_id", col("deportista_id").cast("int")) \
                       .withColumn("genero", col("genero").cast("int")) \
                       .withColumn("edad", col("edad").cast("int")) \
                       .withColumn("altura", col("altura").cast("int")) \
                       .withColumn("peso", col("peso").cast("double")) \
                       .withColumn("equipo_id", col("equipo_id").cast("int"))

deportista.show(5)
deportista.printSchema()

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24|   180|80.0|      199|
|            2|            A Lamusi|     1|  23|   170|60.0|      199|
|            3| Gunnar Nielsen Aaby|     1|  24|     0| 0.0|      273|
|            4|Edgar Lindenau Aabye|     1|  34|     0| 0.0|      278|
|            5|Christine Jacoba ...|     2|  21|   185|82.0|      705|
+-------------+--------------------+------+----+------+----+---------+
only showing top 5 rows

root
 |-- deportista_id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: integer (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)



## 3. RDDs: transformaciones (ID 3.2)

_Pendiente_

## 4. DataFrames: creación e integración (ID 3.1)

_Pendiente_

## 5. Paralelismo (ID 3.4)

_Pendiente_

## 6. Inspección del DataFrame (ID 3.1)

_Pendiente_

## 7. Columnas calculadas

_Pendiente_

## 8. Agregaciones requeridas (ID 3.3)

_Pendiente_